In [ ]:
# ###########################################################################
# REPLACE CELL 2 AND CELL 5 (again -- sorry).
#
# 1. KeyError 'Donald Trump': re-running cell 5 after an interrupt left the
#    old producer thread alive, writing stale results into the new run's
#    queue. The queue and job list are now passed in as arguments, and the
#    consumer ignores items for people it is not tracking.
#
# 2. 62-hour ETA: it was downloading full-resolution originals -- 3.5 MB
#    average, up to 30 MB, about 480 GB for the run. Now fetches an 800px
#    copy via Special:FilePath, roughly 35 GB, so about 2 hours. Built from
#    the title, so your discovery cache stays valid. The last retry still
#    falls back to the original if the thumbnail will not come.
#
# RESTART THE RUNTIME FIRST (Runtime > Restart session) to kill the zombie
# producer thread, then run the cells in order.
# ###########################################################################


In [ ]:
# ==== CELL 2 - config ======================================================
import os

from google.colab import drive

drive.mount("/content/drive")

WORK = "/content/drive/MyDrive/Projects/doppelmap"
PEOPLE_FILE = f"{WORK}/consistent_people.json"   # <-- your pageview-ranked list
OUT_DIR = f"{WORK}/faces"                     # one JSON per person lands here
DISCOVERY_CACHE = f"{WORK}/discovery_cache.json"
os.makedirs(OUT_DIR, exist_ok=True)

# Wikimedia blocks generic user agents. A descriptive one with contact info is
# required by their policy, and is the difference between this working and
# getting 403ed a few thousand requests in.
USER_AGENT = "doppelmap/1.0 (https://github.com/Gandagorn/doppelmap)"

MAX_IMAGES_PER_PERSON = 40
MAX_FACES_PER_IMAGE = 4       # group shots: keep the largest few, not all
MIN_DET_SCORE = 0.60          # detector confidence floor
MIN_FACE_PX = 60              # faces smaller than this embed poorly
DOWNLOAD_WORKERS = 16         # image fetches from upload.wikimedia.org
# The *API* is rate limited far more tightly than the image servers, and
# Wikimedia asks for serial requests against it. Sixteen threads hammering
# search got the whole run 429ed, which silently produced empty results for
# people like Bruce Willis and Dua Lipa. Keep this low.
DISCOVERY_WORKERS = 3
API_DELAY = 0.35              # seconds between API calls, per thread
QUEUE_DEPTH = 64              # decoded images awaiting the GPU (RAM bound)
REQUEST_TIMEOUT = 20          # seconds per HTTP request
MAX_RETRIES = 5               # image downloads, incl. 429 backoff
API_RETRIES = 6               # API calls, which hit 429s and need patience
MAX_IMAGE_BYTES = 25_000_000  # skip absurd originals
# Download a resized copy, not the original. Commons originals average
# 3.5 MB and reach 30 MB, which is ~480 GB across a full run and turns a
# two-hour job into a two-day one. Faces only need enough pixels to clear
# MIN_FACE_PX, so this loses nothing that matters.
THUMB_WIDTH = 800

print("output ->", OUT_DIR)


In [ ]:
# ==== CELL 5 - download (threads) + embed (GPU) ============================
import queue
import urllib.parse

import cv2
import numpy as np
from insightface.app import FaceAnalysis
from insightface.utils import face_align

app = FaceAnalysis(
    name="buffalo_l",
    allowed_modules=["detection", "recognition"],
    providers=["CUDAExecutionProvider"],
)
app.prepare(ctx_id=0, det_size=(640, 640))
recognizer = app.models["recognition"]


def embed_with_flip(img, face):
    """Embedding averaged over the face and its mirror image.

    Test-time flip augmentation. ArcFace is sensitive to which way a head
    is turned and to light falling on one side, and averaging a face with
    its mirror cancels part of that, giving a slightly steadier vector for
    the same person across photos.

    Costs one extra forward pass on a 112x112 crop, which is nothing next
    to detection -- both views go through as a single batch.
    """
    aligned = face_align.norm_crop(img, landmark=face.kps, image_size=112)
    feats = np.asarray(recognizer.get_feat([aligned, cv2.flip(aligned, 1)]))
    summed = feats.reshape(2, -1).sum(axis=0)
    return summed / np.linalg.norm(summed)


def thumb_url(rec, width=THUMB_WIDTH):
    """A resized copy of a Commons image, via Special:FilePath.

    Built from the title, which every cached record already has, so
    switching to thumbnails costs no re-discovery. Deriving the
    upload.wikimedia.org /thumb/ path directly does not work: the allowed
    widths are a per-image bucket list and anything else returns HTTP 400.
    Special:FilePath negotiates that server-side and redirects to the
    nearest valid size.
    """
    title = rec.get("title", "")
    if ":" not in title:
        return rec["original_url"]
    name = urllib.parse.quote(title.split(":", 1)[1].replace(" ", "_"))
    return f"https://commons.wikimedia.org/wiki/Special:FilePath/{name}?width={width}"


def fetch(job):
    """Download and decode one image. Runs on a download thread.

    Handles 429 properly. The first version treated a rate-limited reply as
    an ordinary failure and gave up after three one-second retries, so under
    throttling most images were quietly dropped and people ended up with a
    fraction of their photos -- or none at all, which then looked exactly
    like "this person has no usable pictures".
    """
    person, rec = job
    for attempt in range(MAX_RETRIES):
        try:
            url = thumb_url(rec) if attempt < MAX_RETRIES - 1 else rec["original_url"]
            r = session().get(url, timeout=REQUEST_TIMEOUT)
            if r.status_code == 429:
                wait = float(r.headers.get("Retry-After") or 0) or min(60, 5 * 2 ** attempt)
                time.sleep(wait)
                continue
            if r.ok and r.content:
                img = cv2.imdecode(np.frombuffer(r.content, np.uint8), cv2.IMREAD_COLOR)
                return person, rec, img
            if r.status_code < 500:
                return person, rec, None      # 404 etc: retrying will not help
        except requests.RequestException:
            pass
        time.sleep(min(30, 2 * 2 ** attempt))
    return person, rec, None


def faces_in(img):
    """Every usable face in one image, largest first.

    All of them, not just the most central one. A correctly-titled photo is
    often a group shot, and choosing a face here would be guessing; keeping
    them all lets the later consensus step decide which face is actually
    this person, using the evidence of their other photos.
    """
    found = []
    for f in app.get(img):
        if f.det_score < MIN_DET_SCORE:
            continue
        x1, y1, x2, y2 = (float(v) for v in f.bbox)
        if min(x2 - x1, y2 - y1) < MIN_FACE_PX:
            continue
        if f.kps is None:          # no landmarks, so no reliable alignment
            continue
        found.append({
            "bbox": [round(v, 1) for v in (x1, y1, x2, y2)],
            "det_score": round(float(f.det_score), 3),
            # Rounded to 4 decimals: measured on real embeddings that shifts
            # cosine similarity by at most 1.7e-04, far below anything that
            # could reorder a ranking, while cutting each stored vector from
            # ~13.7 KB to ~3.5 KB. Across 5,000 people that is the difference
            # between roughly 2 GB and 500 MB on Drive.
            "emb": [round(v, 4) for v in embed_with_flip(img, f).astype(float).tolist()],
        })
    found.sort(key=lambda d: -(d["bbox"][2] - d["bbox"][0]) * (d["bbox"][3] - d["bbox"][1]))
    return found[:MAX_FACES_PER_IMAGE]


jobs = [(p, rec) for p, recs in candidates.items() for rec in recs]
results = {p: [] for p in candidates}
pending = {p: len(recs) for p, recs in candidates.items()}
stats = {"ok": 0, "no_face": 0, "failed": 0, "faces": 0}
# `failed` counts downloads that never produced an image. A large value
# means throttling or dead URLs, not people without photos -- worth knowing
# before a thin dataset is mistaken for a thin subject.


def flush(person):
    """Write one person's file as soon as their last image is processed.

    Per-person checkpointing is what makes a Colab disconnect survivable:
    CELL 4 skips anyone whose file already exists, so re-running picks up
    where this left off. The temp-then-rename keeps a killed runtime from
    leaving a half-written file that would then be skipped as "done".
    """
    path = f"{OUT_DIR}/{person}.json"
    payload = {
        "name": person,
        "views": VIEWS.get(person),
        "collected_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "images": results.get(person, []),
    }
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as fh:
        json.dump(payload, fh, ensure_ascii=False)
    os.replace(tmp, path)
    results.pop(person, None)


work = queue.Queue(maxsize=QUEUE_DEPTH)


def producer(out_queue, todo):
    """Download in parallel, feeding the GPU thread through `out_queue`.

    The queue and job list are arguments, not globals. They used to be
    looked up globally, so re-running this cell after an interrupt left the
    previous run's producer alive and writing its stale results into the
    new run's queue -- which surfaced as a KeyError on a person the new run
    had already finished and flushed.
    """
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
        for item in pool.map(fetch, todo):
            out_queue.put(item)   # blocks when full, throttling to GPU speed
    out_queue.put(None)


threading.Thread(target=producer, args=(work, jobs), daemon=True).start()

with tqdm(total=len(jobs), desc="embed") as bar:
    while True:
        item = work.get()
        if item is None:
            break
        person, rec, img = item
        if person not in pending:
            continue              # stale item from an earlier interrupted run
        if img is None:
            stats["failed"] += 1
        else:
            found = faces_in(img)
            if found:
                results[person].append({**rec, "faces": found})
                stats["ok"] += 1
                stats["faces"] += len(found)
            else:
                stats["no_face"] += 1
        pending[person] -= 1
        if pending[person] == 0:
            flush(person)
        bar.update(1)

for person in list(results):      # anyone with zero candidate images
    flush(person)

print(stats)
